# Real-Time Social Media Sentiment Monitoring
## Notebook 1 — Data exploration, NLP preprocessing and Spark MLlib model training

**Subject:** Big Data Analytics / Scalable Machine Learning  
**Stack:** Apache Spark 3.5 · Spark MLlib · Kafka · MongoDB · FastAPI · Streamlit

This notebook is the **ML development** stage of the pipeline:

`Real dataset → EDA → NLP cleaning → TF-IDF → Spark MLlib (LR / NB / RF) → evaluation → saved model`

The saved model is then consumed by `Spark/streaming.py` (Structured Streaming) and by the
FastAPI prediction engine.


## Step 1 — Install & import libraries

In [ ]:
# Google Colab only
!pip -q install pyspark==3.5.1 nltk wordcloud langdetect seaborn

In [ ]:
import os, re, time, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression, NaiveBayes, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

spark = (SparkSession.builder
         .appName("sentiment-training-notebook")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.shuffle.partitions", "64")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## Step 2 — Load the real dataset

We use **Sentiment140** (1.6M real tweets) as the primary dataset and **Amazon product
reviews** as the secondary dataset (it supplies the *neutral* class from 3-star reviews).

```bash
kaggle datasets download -d kazanova/sentiment140 -p Dataset/
unzip Dataset/sentiment140.zip -d Dataset/
```
See `Dataset/README.md` for full download instructions.

In [ ]:
TWITTER_CSV = "Dataset/twitter_sentiment.csv"   # training.1600000.processed.noemoticon.csv
AMAZON_JSON = "Dataset/reviews_Electronics_5.json"  # optional, gives the neutral class

schema = T.StructType([
    T.StructField("target", T.StringType()), T.StructField("ids", T.StringType()),
    T.StructField("date", T.StringType()),   T.StructField("flag", T.StringType()),
    T.StructField("user", T.StringType()),   T.StructField("text", T.StringType())])

tw = (spark.read.csv(TWITTER_CSV, schema=schema, header=False, quote='"', escape='"')
      .withColumn("label_str", F.when(F.col("target")=="0","negative")
                                .when(F.col("target")=="4","positive")
                                .otherwise("neutral"))
      .withColumn("source", F.lit("twitter"))
      .select("text","label_str","source", F.col("date").alias("timestamp")))

if os.path.exists(AMAZON_JSON):
    az = spark.read.json(AMAZON_JSON)
    az = (az.withColumn("label_str", F.when(F.col("overall")<=2,"negative")
                                      .when(F.col("overall")==3,"neutral")
                                      .otherwise("positive"))
            .withColumn("source", F.lit("amazon"))
            .select(F.col("reviewText").alias("text"),"label_str","source",
                    F.col("unixReviewTime").cast("string").alias("timestamp")))
    df = tw.unionByName(az)
else:
    df = tw

df.cache()
print("Total records:", f"{df.count():,}")
df.show(5, truncate=90)

## Step 3 — Exploratory Data Analysis

In [ ]:
print("Dataset size on disk: %.1f MB" % (os.path.getsize(TWITTER_CSV)/1024/1024))
print("Records              :", f"{df.count():,}")
print("Missing / empty text :", df.filter(F.col("text").isNull() | (F.trim("text")=="")).count())
print("Duplicate texts      :", df.count() - df.select("text").distinct().count())
dist = df.groupBy("label_str").count().toPandas()
display(dist)

In [ ]:
# Graph 1 - sentiment distribution
plt.figure(figsize=(6,4))
sns.barplot(data=dist, x="label_str", y="count",
            palette={"negative":"#dc2626","neutral":"#64748b","positive":"#16a34a"})
plt.title("Sentiment distribution"); plt.ylabel("records"); plt.show()

In [ ]:
# Graph 2 - text length analysis (sampled for plotting)
samp = df.sample(False, 0.02, seed=42).select("text","label_str").toPandas()
samp["chars"] = samp.text.astype(str).str.len()
samp["words"] = samp.text.astype(str).str.split().str.len()

fig, ax = plt.subplots(1,2, figsize=(13,4))
sns.histplot(samp.chars, bins=50, ax=ax[0]).set(title="Character length")
sns.boxplot(data=samp, x="label_str", y="words", ax=ax[1]).set(title="Word count by sentiment")
plt.tight_layout(); plt.show()

In [ ]:
# Graph 3 - word frequency + word cloud
from collections import Counter
from wordcloud import WordCloud
sw = set(stopwords.words("english"))
tokens = [w for t in samp.text.astype(str) for w in re.findall(r"[a-z']{3,}", t.lower()) if w not in sw]
top = Counter(tokens).most_common(25)
plt.figure(figsize=(11,4))
sns.barplot(x=[c for _,c in top], y=[w for w,_ in top]); plt.title("Top 25 words"); plt.show()

wc = WordCloud(width=1100, height=420, background_color="white").generate(" ".join(tokens))
plt.figure(figsize=(12,4.5)); plt.imshow(wc); plt.axis("off"); plt.show()

## Step 4 — Text preprocessing (NLP)

Steps applied: lowercase → remove URLs → remove @mentions → remove #hashtags →
remove punctuation/emoji → tokenize → remove stop words → lemmatize.

```
Input : "I love this new phone!!! 🔥"
Output: "love new phone"
```

In [ ]:
lemmatizer = WordNetLemmatizer()
STOP = set(stopwords.words("english"))

def preprocess(text: str) -> str:
    t = str(text).lower()
    t = re.sub(r"http\S+|www\.\S+", " ", t)      # URLs
    t = re.sub(r"@\w+", " ", t)                    # mentions
    t = re.sub(r"#(\w+)", " ", t)                  # hashtags
    t = re.sub(r"[^a-z\u0900-\u097F\s]", " ", t)  # punctuation + emoji (keep Devanagari)
    tokens = word_tokenize(t)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in STOP and len(w) > 2]
    return " ".join(tokens)

print(repr(preprocess("I love this new phone!!! 🔥")))
print(repr(preprocess("@amazon the delivery was HORRIBLE http://t.co/x #fail")))

In [ ]:
# Distributed version: the same cleaning expressed as native Spark column operations
# (no Python UDF -> no serialization overhead, runs across all executors)
def clean_col(col):
    c = F.lower(col)
    c = F.regexp_replace(c, r"http\S+|www\.\S+", " ")
    c = F.regexp_replace(c, r"@\w+", " ")
    c = F.regexp_replace(c, r"#\w+", " ")
    c = F.regexp_replace(c, r"[^a-z\u0900-\u097F\s]", " ")
    return F.trim(F.regexp_replace(c, r"\s+", " "))

clean = (df.filter(F.col("text").isNotNull())
           .withColumn("clean_text", clean_col(F.col("text")))
           .filter(F.length("clean_text") > 5)
           .dropDuplicates(["clean_text"])
           .withColumn("label", F.when(F.col("label_str")=="negative",0.0)
                                 .when(F.col("label_str")=="neutral",1.0).otherwise(2.0))
           .select("clean_text","label","label_str").cache())
print("After cleaning:", f"{clean.count():,}")
clean.show(5, truncate=80)

## Performance check — plain Python vs Apache Spark

Why Spark instead of pandas: the cleaning is embarrassingly parallel, but pandas runs it in a
single process on one core and must hold the whole frame in RAM. Spark partitions the data,
runs the same transformation on every core/executor, and never materialises the full dataset.

In [ ]:
N = 300_000
t0 = time.time()
pdf = pd.read_csv(TWITTER_CSV, encoding="latin-1", header=None, nrows=N,
                  names=["target","ids","date","flag","user","text"])
pdf["clean"] = pdf.text.astype(str).map(lambda s: " ".join(
    re.sub(r"[^a-z\s]", " ", re.sub(r"http\S+|@\w+|#\w+", " ", s.lower())).split()))
pdf = pdf.drop_duplicates("clean")
py_time = time.time() - t0

t0 = time.time()
sdf = (spark.read.csv(TWITTER_CSV, schema=schema, header=False).limit(N)
       .withColumn("clean", clean_col(F.col("text"))).dropDuplicates(["clean"]))
_ = sdf.count()
sp_time = time.time() - t0

print(f"Python (pandas, 1 core): {py_time:6.2f}s")
print(f"Spark  (distributed)   : {sp_time:6.2f}s")
print(f"Speed-up               : {py_time/sp_time:.2f}x")

## Step 5 — Feature extraction (TF-IDF)

In [ ]:
tokenizer = Tokenizer(inputCol="clean_text", outputCol="words")
hashing   = HashingTF(inputCol="words", outputCol="tf", numFeatures=1<<18)
idf       = IDF(inputCol="tf", outputCol="features", minDocFreq=3)

train, test = clean.randomSplit([0.8, 0.2], seed=42)
print(f"train={train.count():,}  test={test.count():,}")

## Step 6 — Machine learning with Spark MLlib (3 models)

In [ ]:
def build(clf):
    return Pipeline(stages=[tokenizer, hashing, idf, clf])

models = {
  "Logistic Regression": LogisticRegression(maxIter=25, regParam=0.01),
  "Naive Bayes":         NaiveBayes(smoothing=1.0, modelType="multinomial"),
  "Random Forest":       RandomForestClassifier(numTrees=60, maxDepth=12, seed=42),
}

fitted, preds = {}, {}
for name, clf in models.items():
    t0 = time.time()
    fitted[name] = build(clf).fit(train)
    preds[name]  = fitted[name].transform(test).cache()
    print(f"{name:22s} trained in {time.time()-t0:6.1f}s")

## Step 7 — Evaluation: accuracy, precision, recall, F1, confusion matrix, ROC

In [ ]:
def metrics(pred):
    out = {}
    for m in ["accuracy","weightedPrecision","weightedRecall","f1"]:
        out[m] = MulticlassClassificationEvaluator(labelCol="label",
                    predictionCol="prediction", metricName=m).evaluate(pred)
    return out

results = pd.DataFrame({n: metrics(p) for n, p in preds.items()}).T
results.columns = ["Accuracy","Precision","Recall","F1"]
display(results.round(4))
results.plot.bar(figsize=(9,4), rot=0, title="Spark MLlib model comparison"); plt.show()

In [ ]:
best_name = results["F1"].idxmax()
print("Best model:", best_name)
best_pred = preds[best_name]

labels = ["negative","neutral","positive"]
cm = (best_pred.groupBy("label","prediction").count().toPandas()
      .pivot(index="label", columns="prediction", values="count").fillna(0))
plt.figure(figsize=(5.5,4.5))
sns.heatmap(cm, annot=True, fmt=".0f", cmap="Blues",
            xticklabels=labels[:cm.shape[1]], yticklabels=labels[:cm.shape[0]])
plt.title(f"Confusion matrix — {best_name}"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

In [ ]:
# ROC curve (one-vs-rest: positive vs rest)
from sklearn.metrics import roc_curve, auc
pdf_roc = (best_pred.select("label","probability").sample(False, 0.2, seed=1)
           .rdd.map(lambda r: (1 if r["label"]==2.0 else 0, float(r["probability"][2]))).collect())
y, s = zip(*pdf_roc)
fpr, tpr, _ = roc_curve(y, s)
plt.figure(figsize=(5.5,4.5))
plt.plot(fpr, tpr, label=f"AUC = {auc(fpr,tpr):.3f}")
plt.plot([0,1],[0,1],"--",c="grey"); plt.legend()
plt.title("ROC curve — positive vs rest"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.show()

## Step 8 — Save the best model + metrics (consumed by streaming & API)

In [ ]:
os.makedirs("Models", exist_ok=True)
fitted[best_name].write().overwrite().save("Models/best_model")

payload = {"best_model": best_name, "labels": labels,
           "models": {n: {"accuracy":m["accuracy"], "weightedPrecision":m["weightedPrecision"],
                          "weightedRecall":m["weightedRecall"], "f1":m["f1"],
                          "confusion_matrix": [[int(r.label), int(r.prediction), r["count"]]
                             for r in preds[n].groupBy("label","prediction").count().collect()]}
                      for n, m in ((n, metrics(p)) for n, p in preds.items())},
           "trained_at": time.strftime("%Y-%m-%dT%H:%M:%S")}
json.dump(payload, open("Models/metrics.json","w"), indent=2)
print("Saved Models/best_model and Models/metrics.json")

## Step 9 — Prediction demo (the exact contract served by `POST /predict`)

In [ ]:
import datetime as dt
def predict(text):
    row = fitted[best_name].transform(
        spark.createDataFrame([(preprocess(text),)], ["clean_text"])
    ).select("prediction","probability").head()
    return {"text": text,
            "sentiment": labels[int(row["prediction"])],
            "confidence": round(float(max(row["probability"])), 2),
            "timestamp": dt.datetime.utcnow().isoformat()}

for t in ["The product quality is amazing",
          "Worst service ever, totally disappointed",
          "The package arrived on Tuesday"]:
    print(json.dumps(predict(t), indent=2))

## Step 10 — Optional: export a lightweight scikit-learn twin for the API container

The Spark model stays the project's training artefact; this joblib export lets the FastAPI
container answer `/predict` in milliseconds without booting a JVM.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression as SkLR
import joblib

sample = clean.sample(False, 0.15, seed=7).select("clean_text","label").toPandas()
sk = make_pipeline(TfidfVectorizer(max_features=200_000, ngram_range=(1,2)),
                   SkLR(max_iter=1000, n_jobs=-1))
sk.fit(sample.clean_text, sample.label.astype(int))
joblib.dump(sk, "Models/sklearn_model.joblib")
print("saved Models/sklearn_model.joblib")